In [1]:
import dvc 
import pandas as pd
import polars as pl
from pathlib import Path
import duckdb

In [2]:
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "raw" / "batch_readings.parquet"
print(BASE_DIR)

c:\Users\Khadija Ismail\Desktop\claude_projects\batch_reactor_project_one


In [3]:
# using parquet
parquet_dataframe = pl.read_parquet(DATA_PATH)
parquet_dataframe.head()

batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
i64,datetime[μs],str,f64,f64,f64,f64,f64,str,f64,str,f64,f64,str,str
0,2022-01-01 00:52:00,"""Reactor 5""",24.688933,0.988253,117.363277,50.401075,0.963375,"""L119""",6.585012,"""A""",54.050379,59.521627,"""degC""","""bar"""
0,2022-01-01 00:53:00,"""R5""",78.129361,0.989035,110.259602,50.823903,0.963375,"""L119""",6.652077,"""A""",54.050379,59.521627,"""F""",null
0,2022-01-01 00:54:00,"""R5""",29.354209,1.00185,113.898923,51.681363,0.963375,"""L119""",6.668752,"""A""",54.050379,59.521627,"""c""","""BAR"""
0,2022-01-01 00:55:00,"""R5""",32.377277,1.115178,115.309385,51.026732,0.963375,"""L119""",6.590493,"""A""",54.050379,59.521627,"""C""",null
0,2022-01-01 00:56:00,"""R5""",32.307078,1.125136,123.293666,50.495019,0.963375,"""L119""",6.609744,"""A """,54.050379,59.521627,"""degC""","""BAR"""


In [ ]:
len(parquet_dataframe['reactor_id'].unique())

In [ ]:
len(parquet_dataframe['operator_shift'].unique())

In [ ]:
len(parquet_dataframe["catalyst_lot"].unique())

In [25]:
# using pandas 
pandas_dataframe = pd.read_parquet(DATA_PATH)
pandas_dataframe.head()

,batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
0,0,2022-01-01 00:52:00,Reactor 5,24.688933,0.988253,117.363277,50.401075,0.963375,L119,6.585012,A,54.050379,59.521627,degC,bar
1,0,2022-01-01 00:53:00,R5,78.129361,0.989035,110.259602,50.823903,0.963375,L119,6.652077,A,54.050379,59.521627,F,NaN
2,0,2022-01-01 00:54:00,R5,29.354209,1.001850,113.898923,51.681363,0.963375,L119,6.668752,A,54.050379,59.521627,c,BAR
3,0,2022-01-01 00:55:00,R5,32.377277,1.115178,115.309385,51.026732,0.963375,L119,6.590493,A,54.050379,59.521627,C,NaN
4,0,2022-01-01 00:56:00,R5,32.307078,1.125136,123.293666,50.495019,0.963375,L119,6.609744,A,54.050379,59.521627,degC,BAR


In [8]:
pandas_dataframe.shape

(2009978, 15)

In [4]:
# using duck db 
conn = duckdb.connect()
duck_data = conn.execute(f"""
SELECT reactor_id, operator_shift,
COUNT(*) AS row_count,
AVG(temp_c) AS avg_temperature,
MAX(temp_c) AS max_temperature,
AVG(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) AS avg_null_temp 
FROM '{DATA_PATH}'
GROUP BY reactor_id, operator_shift
""").df()

display(duck_data.head(10))

,reactor_id,operator_shift,row_count,avg_temperature,max_temperature,avg_null_temp
0,r8,Night,1479,85.155344,363.414102,0.041244
1,Reactor 4,B,1387,86.710568,366.665823,0.038933
2,R4,N,4652,86.331389,533.455740,0.052666
3,Reactor 4,Night,177,81.762382,367.044135,0.056497
4,Reactor 4,night,172,84.816883,353.010613,0.058140
5,r6,C,1712,86.097874,364.516159,0.047897
6,R-6,C,1711,87.719587,392.935232,0.037989
7,R-8,C,1530,84.711057,362.044125,0.048366
8,Reactor 8,Shift C,253,85.093264,351.996259,0.090909
9,R1,Night,4690,86.328155,367.721275,0.052878


In [ ]:
# select distict reactor_id, operation ship and units 
distinct_values = conn.execute(f"""
SELECT DISTINCT reactor_id, operator_shift, pressure_unit, temp_unit
FROM '{DATA_PATH}'
""").df()
print(distinct_values.head(20))

In [ ]:
scan_dataframe = pl.scan_parquet(DATA_PATH)
scan_dataframe

In [ ]:
# lets first identify the naming mimatched
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
# check the unique values of the pressure unit and temperature unit 
parquet_dataframe['pressure_unit'].unique()

In [ ]:
parquet_dataframe['temp_unit'].unique()

In [17]:
parquet_dataframe.describe()

statistic,batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
str,f64,str,str,f64,f64,f64,f64,f64,str,f64,str,f64,f64,str,str
"""count""",2.009978e6,"""2009978""","""2009978""",1.920603e6,1.923068e6,1.932718e6,1.92356e6,2.009978e6,"""2009978""",1.924686e6,"""2009978""",2.009978e6,2.009978e6,"""1607917""","""1607012"""
"""null_count""",0.0,"""0""","""0""",89375.0,86910.0,77260.0,86418.0,0.0,"""0""",85292.0,"""0""",0.0,0.0,"""402061""","""402966"""
"""mean""",5555.049115,"""2022-01-28 02:20:38.965531""",null,85.526632,12.659136,124.453465,49.984466,4.695325,null,6.914706,null,55.057487,62.299918,null,null
"""std""",3207.614006,null,null,46.896676,39.021974,12.566525,2.081098,18.551248,null,0.235943,null,14.661375,16.665868,null,null
"""min""",0.0,"""2022-01-01 00:27:00""","""R-1""",-1237.866865,-1136.653809,-268.501611,41.195497,0.912991,"""L000""",5.869324,"""A""",22.562324,0.0,"""C""","""BAR"""
"""25%""",2777.0,"""2022-01-14 14:15:00""",null,71.587952,1.705644,116.738545,48.56554,0.960358,null,6.755873,null,41.659302,51.867891,null,null
"""50%""",5555.0,"""2022-01-28 02:17:00""",null,78.832267,1.817283,121.317067,49.99773,0.970738,null,6.915301,null,55.213736,64.462795,null,null
"""75%""",8333.0,"""2022-02-10 14:24:00""",null,84.235437,1.916053,127.078896,51.388529,0.981252,null,7.073749,null,68.47019,75.120362,null,null
"""max""",11110.0,"""2022-02-24 04:40:00""","""r8""",1281.549996,1092.143043,490.414392,59.035988,100.0,"""l119""",7.927774,"""night""",87.454313,97.302718,"""°F""","""psig"""


In [ ]:
parquet_dataframe.dtypes

In [ ]:
# check for duplicated columns 
parquet_dataframe.is_duplicated().sum()

In [ ]:
# get the average temperature in pandas 
pandas_dataframe.info()

In [ ]:
# check distinct batches i have 
batches_data = conn.execute(f"""
SELECT DISTINCT(batch_id) AS unique_batches, COUNT(*) AS counts_per_batch,
AVG(temp_c) AS average_temperature_per_batch,
AVG(pressure_bar) AS average_pressure_per_batch,
AVG(ph) as avg_ph_per_batch
FROM '{DATA_PATH}'
GROUP BY batch_id
ORDER BY batch_id ASC
""").df()
display(batches_data.head(20))

In [ ]:
# check distinct batches i have 
catalyst_lot = conn.execute(f"""
SELECT DISTINCT(catalyst_lot)
FROM '{DATA_PATH}'
GROUP BY catalyst_lot
ORDER BY catalyst_lot ASC
""").df()
display(catalyst_lot.tail(20))

In [ ]:
len(parquet_dataframe['catalyst_lot'].unique())

In [ ]:
# inspect the negetive values in the agitation_rpm, temp_c and pressure_bar 
negetive_values = conn.execute(f"""
SELECT agitator_rpm, temp_c, pressure_bar, pressure_unit, temp_unit
FROM '{DATA_PATH}'
WHERE agitator_rpm <= 0 OR temp_c <= 0 OR pressure_bar <= 0
""").df()
display(negetive_values.head(10))
print(len(negetive_values))

In [ ]:
# lets check the negetive values independently 
negetive_temp = conn.execute(f"""SELECT temp_c, temp_unit FROM '{DATA_PATH}' WHERE temp_c <= 0""").df()
display(negetive_temp.sample(20))
print(len(negetive_temp))

In [ ]:
# negetive agitator_rpm
negetive_agitator_rpm = conn.execute(f"""SELECT agitator_rpm FROM '{DATA_PATH}' WHERE agitator_rpm <= 0""").df()
display(negetive_agitator_rpm.sample(20))
print(len(negetive_agitator_rpm))

In [ ]:
negetive_pressure = conn.execute(f"""SELECT pressure_bar, pressure_unit FROM '{DATA_PATH}' WHERE pressure_bar <= 0""").df()
display(negetive_pressure.sample(20))
print(len(negetive_pressure))


In [ ]:
# check the percentage of nulls after grouping by reactor_id and the count of nulls 
null_by_reactor = conn.execute(f"""
SELECT reactor_id, SUM(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) AS temp_missing_count, ROUND(100 * (SUM(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS temp_pct,
SUM(CASE WHEN pressure_bar IS NULL THEN 1 ELSE 0 END) AS pressure_missing_count, ROUND(100 * (SUM(CASE WHEN pressure_bar IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS pressure_pct,
SUM(CASE WHEN agitator_rpm IS NULL THEN 1 ELSE 0 END) AS agitator_missing_count, ROUND(100 * (SUM(CASE WHEN agitator_rpm IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS agitator_pct,
SUM(CASE WHEN feed_rate IS NULL THEN 1 ELSE 0 END) AS feed_rate_missing_count, ROUND(100 * (SUM(CASE WHEN feed_rate IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS feed_rate_pct,
SUM(CASE WHEN ph IS NULL THEN 1 ELSE 0 END) AS ph_missing_count, ROUND(100 * (SUM(CASE WHEN ph IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS ph_pct,
SUM(CASE WHEN temp_unit IS NULL THEN 1 ELSE 0 END) AS temp_unit_missing_count, ROUND(100 * (SUM(CASE WHEN temp_unit IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS temp_unit_pct,
SUM(CASE WHEN pressure_unit IS NULL THEN 1 ELSE 0 END) AS pressure_unit_missing_count, ROUND(100 * (SUM(CASE WHEN pressure_unit IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS pressure_unit_pct
FROM '{DATA_PATH}' 
GROUP BY reactor_id;
""").df()
display(null_by_reactor)

In [5]:
null_by_operator_shift = conn.execute(f"""
SELECT operator_shift, SUM(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) AS temp_missing_count, ROUND(100 * (SUM(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS temp_pct,
SUM(CASE WHEN pressure_bar IS NULL THEN 1 ELSE 0 END) AS pressure_missing_count, ROUND(100 * (SUM(CASE WHEN pressure_bar IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS pressure_pct,
SUM(CASE WHEN agitator_rpm IS NULL THEN 1 ELSE 0 END) AS agitator_missing_count, ROUND(100 * (SUM(CASE WHEN agitator_rpm IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS agitator_pct,
SUM(CASE WHEN feed_rate IS NULL THEN 1 ELSE 0 END) AS feed_rate_missing_count, ROUND(100 * (SUM(CASE WHEN feed_rate IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS feed_rate_pct,
SUM(CASE WHEN ph IS NULL THEN 1 ELSE 0 END) AS ph_missing_count, ROUND(100 * (SUM(CASE WHEN ph IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS ph_pct,
SUM(CASE WHEN temp_unit IS NULL THEN 1 ELSE 0 END) AS temp_unit_missing_count, ROUND(100 * (SUM(CASE WHEN temp_unit IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS temp_unit_pct,
SUM(CASE WHEN pressure_unit IS NULL THEN 1 ELSE 0 END) AS pressure_unit_missing_count, ROUND(100 * (SUM(CASE WHEN pressure_unit IS NULL THEN 1 ELSE 0 END) / COUNT(*)), 2) AS pressure_unit_pct
FROM '{DATA_PATH}' 
GROUP BY operator_shift;
""").df()
display(null_by_operator_shift)

,operator_shift,temp_missing_count,temp_pct,pressure_missing_count,pressure_pct,agitator_missing_count,agitator_pct,feed_rate_missing_count,feed_rate_pct,ph_missing_count,ph_pct,temp_unit_missing_count,temp_unit_pct,pressure_unit_missing_count,pressure_unit_pct
0,night,1869.0,4.39,1752.0,4.12,1588.0,3.73,1806.0,4.24,1785.0,4.19,8514.0,20.01,8569.0,20.14
1,Night,1938.0,4.54,1797.0,4.21,1680.0,3.93,1819.0,4.26,1806.0,4.23,8378.0,19.61,8537.0,19.98
2,B,15664.0,4.27,16225.0,4.42,14190.0,3.87,15677.0,4.27,16096.0,4.39,73711.0,20.09,73418.0,20.01
3,N,1948.0,4.59,1817.0,4.28,1667.0,3.93,1859.0,4.38,1765.0,4.16,8452.0,19.92,8565.0,20.19
4,Shift A,1874.0,4.44,1833.0,4.35,1654.0,3.92,1840.0,4.36,1805.0,4.28,8302.0,19.69,8320.0,19.73
5,A,1907.0,4.46,1829.0,4.27,1638.0,3.83,1896.0,4.43,1811.0,4.23,8667.0,20.26,8616.0,20.14
6,a,1801.0,4.26,1724.0,4.08,1543.0,3.65,1844.0,4.36,1761.0,4.17,8373.0,19.82,8409.0,19.90
7,Shift C,2790.0,4.48,2737.0,4.39,2327.0,3.74,2812.0,4.51,2539.0,4.08,12469.0,20.02,12498.0,20.07
8,c,2762.0,4.41,2671.0,4.27,2395.0,3.83,2684.0,4.29,2730.0,4.36,12492.0,19.95,12572.0,20.08
9,Night,17689.0,4.64,15823.0,4.15,14763.0,3.87,16234.0,4.26,15726.0,4.13,76140.0,19.97,76387.0,20.04


In [ ]:
# check the distinct values of the shift
shift_check = conn.execute(f"""
SELECT DISTINCT(operator_shift) as unique_operator, timestamp 
FROM '{DATA_PATH}'
""").df()
display(shift_check.sample())

,unique_operator,timestamp
0,Night,2022-02-18 17:04:00
1,Night,2022-02-18 17:06:00
2,Night,2022-02-18 17:14:00
3,Night,2022-02-18 15:23:00
4,Night,2022-02-18 15:28:00
...,...,...
678374,A,2022-01-20 13:49:00
678375,N,2022-01-20 13:05:00
678376,Shift B,2022-01-20 11:32:00
678377,Shift B,2022-01-20 11:38:00


In [6]:
# check avg yield pct for each operation 
avg_yield_pct = conn.execute(f"""
SELECT DISTINCT(operator_shift) as unique_shifts, AVG(yield_pct) AS avg_yield_pct
FROM '{DATA_PATH}'
GROUP BY unique_shifts
""").df()

display(avg_yield_pct)

,unique_shifts,avg_yield_pct
0,Night,61.780880
1,Shift B,62.349570
2,A,62.786196
3,Shift C,62.312864
4,a,62.761046
5,B,62.320754
6,A,62.539437
7,N,61.814829
8,night,61.852328
9,Night,61.887791


In [14]:
def change_values(name):
    name = name.lower().strip()
    if "a" in name:
        return "Shift A"
    if "b" in name:
        return "Shift B"
    if "c" in name:
        return "Shift C"
    if "n" in name:
        return "Night"

avg_yield_pct['group_name'] = avg_yield_pct['unique_shifts'].apply(change_values)
group_by_data = avg_yield_pct.groupby("group_name")['avg_yield_pct'].agg(['mean', "max", "min"])
print(group_by_data)

                 mean        max        min
group_name                                 
Night       61.833957  61.887791  61.780880
Shift A     62.713104  62.786196  62.539437
Shift B     62.330353  62.349570  62.320734
Shift C     62.302440  62.325703  62.268753


In [ ]:
# check throughly for duplicates
pandas_dataframe[pandas_dataframe.duplicated()]

,batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
335,1,2022-01-01 03:59:00,R7,80.802279,NaN,109.968834,49.135980,0.962882,L002,6.987468,A,52.904138,69.576301,degC,bar
762,4,2022-01-01 02:39:00,R8,144.778405,23.413755,120.108665,NaN,0.987677,L061,7.051903,Night,47.522894,58.296184,°F,psig
815,4,2022-01-01 03:31:00,R8,72.452756,1.719635,117.152739,52.867226,0.987677,L061,6.899153,Night,47.522894,58.296184,NaN,bar
848,4,2022-01-01 04:03:00,R8,75.882595,1.785791,NaN,53.552216,0.987677,L061,6.908736,N,47.522894,58.296184,NaN,BAR
1046,5,2022-01-01 04:27:00,R8,76.864887,1.788194,123.254375,50.256709,0.972989,L012,6.580548,C,47.519392,28.860210,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009034,11105,2022-02-24 02:34:00,R4,92.864689,2.051296,110.891177,49.416188,0.979077,L079,7.088104,Night,56.449530,16.920549,c,barg
2009041,11105,2022-02-24 02:40:00,R4,94.542142,2.052833,123.331881,49.402857,0.979077,L079,NaN,Night,56.449530,16.920549,C,bar
2009049,11105,2022-02-24 02:47:00,R4,94.467216,NaN,127.719273,49.665339,0.979077,L079,7.062532,Night,56.449530,16.920549,c,NaN
2009154,11106,2022-02-24 02:30:00,R8,86.855247,1.925681,155.732843,49.849485,0.955476,L033,7.527923,Shift C,55.170213,81.626458,degC,barg


In [12]:
all_duplicated_row_counts = pandas_dataframe[pandas_dataframe.duplicated(keep=False)]
all_duplicated_row_counts

,batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
334,1,2022-01-01 03:59:00,R7,80.802279,NaN,109.968834,49.135980,0.962882,L002,6.987468,A,52.904138,69.576301,degC,bar
335,1,2022-01-01 03:59:00,R7,80.802279,NaN,109.968834,49.135980,0.962882,L002,6.987468,A,52.904138,69.576301,degC,bar
761,4,2022-01-01 02:39:00,R8,144.778405,23.413755,120.108665,NaN,0.987677,L061,7.051903,Night,47.522894,58.296184,°F,psig
762,4,2022-01-01 02:39:00,R8,144.778405,23.413755,120.108665,NaN,0.987677,L061,7.051903,Night,47.522894,58.296184,°F,psig
814,4,2022-01-01 03:31:00,R8,72.452756,1.719635,117.152739,52.867226,0.987677,L061,6.899153,Night,47.522894,58.296184,NaN,bar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009049,11105,2022-02-24 02:47:00,R4,94.467216,NaN,127.719273,49.665339,0.979077,L079,7.062532,Night,56.449530,16.920549,c,NaN
2009153,11106,2022-02-24 02:30:00,R8,86.855247,1.925681,155.732843,49.849485,0.955476,L033,7.527923,Shift C,55.170213,81.626458,degC,barg
2009154,11106,2022-02-24 02:30:00,R8,86.855247,1.925681,155.732843,49.849485,0.955476,L033,7.527923,Shift C,55.170213,81.626458,degC,barg
2009201,11106,2022-02-24 03:17:00,R8,86.927621,NaN,165.713876,49.839073,0.955476,L033,7.411849,C,55.170213,81.626458,NaN,NaN


In [19]:
# the distich batches that i have 
distinct_values_batches = pandas_dataframe['batch_id'].unique()
distinct_values_batches

array([    0,     1,     2, ..., 11108, 11109, 11110], shape=(11111,))

In [30]:
# check how the yieid average varies by batch 
average_by_batch = conn.execute(f"""
SELECT batch_id, AVG(yield_pct) AS avg_yield_per_batch
FROM '{DATA_PATH}'
GROUP BY batch_id 
ORDER BY batch_id ASC
""").df()
average_by_batch.head(20)

,batch_id,avg_yield_per_batch
0,0,59.521627
1,1,69.576301
2,2,54.799699
3,3,43.799610
4,4,58.296184
5,5,28.860210
6,6,82.188199
7,7,37.371336
8,8,49.370792
9,9,77.979663


In [33]:
pandas_dataframe.head(10)

,batch_id,timestamp,reactor_id,temp_c,pressure_bar,agitator_rpm,feed_rate,feed_purity,catalyst_lot,ph,operator_shift,ambient_humidity,yield_pct,temp_unit,pressure_unit
0,0,2022-01-01 00:52:00,Reactor 5,24.688933,0.988253,117.363277,50.401075,0.963375,L119,6.585012,A,54.050379,59.521627,degC,bar
1,0,2022-01-01 00:53:00,R5,78.129361,0.989035,110.259602,50.823903,0.963375,L119,6.652077,A,54.050379,59.521627,F,NaN
2,0,2022-01-01 00:54:00,R5,29.354209,1.001850,113.898923,51.681363,0.963375,L119,6.668752,A,54.050379,59.521627,c,BAR
3,0,2022-01-01 00:55:00,R5,32.377277,1.115178,115.309385,51.026732,0.963375,L119,6.590493,A,54.050379,59.521627,C,NaN
4,0,2022-01-01 00:56:00,R5,32.307078,1.125136,123.293666,50.495019,0.963375,L119,6.609744,A,54.050379,59.521627,degC,BAR
5,0,2022-01-01 00:57:00,r5,33.076098,1.148186,126.507262,50.699760,0.963375,L119,6.657038,A,54.050379,59.521627,°C,bar
6,0,2022-01-01 00:58:00,Reactor 5,33.164814,1.104675,116.574943,49.926178,0.963375,L119,6.613868,A,54.050379,59.521627,NaN,barg
7,0,2022-01-01 00:59:00,R5,35.399471,1.140843,118.525533,50.226931,0.963375,L119,6.665979,A,54.050379,59.521627,c,NaN
8,0,2022-01-01 01:00:00,R5,39.497618,17.915759,121.010352,50.303870,0.963375,L119,6.585807,A,54.050379,59.521627,°C,psi
9,0,2022-01-01 01:01:00,R5,38.891914,1.235696,111.388008,50.382383,0.963375,L119,6.561605,a,54.050379,59.521627,c,BAR


In [12]:
# lets check the lot count column
catalyst_lot_count = conn.execute(f"""
SELECT DISTINCT(catalyst_lot) as unique_lot_counts FROM '{DATA_PATH}' GROUP BY catalyst_lot ORDER BY catalyst_lot
""").df()
print(catalyst_lot_count)

    unique_lot_counts
0                L000
1                L001
2                L002
3                L003
4                L004
..                ...
355              l115
356              l116
357              l117
358              l118
359              l119

[360 rows x 1 columns]


In [16]:
# determine if pressure units are constant within batch 
batch_distinct_unit = conn.execute(f"""
SELECT batch_id, COUNT(DISTINCT pressure_unit) as distict_pressure_unit_count, COUNT(DISTINCT temp_unit) AS distinct_temp_unit_count
FROM '{DATA_PATH}'
GROUP BY batch_id
""").df()
display(batch_distinct_unit)

,batch_id,distict_pressure_unit_count,distinct_temp_unit_count
0,10011,8,9
1,10013,8,7
2,10021,8,9
3,10023,8,9
4,10029,8,9
...,...,...,...
11106,5728,8,9
11107,5759,8,7
11108,5944,8,9
11109,5948,8,9


In [20]:
# check the agitation_rpm 
agitation_rpm = conn.execute(f"""
SELECT agitator_rpm FROM '{DATA_PATH}' WHERE agitator_rpm BETWEEN -2 AND 2
""").df()
display(agitation_rpm)

,agitator_rpm
0,-1.226078
1,0.177013
2,-0.458265
3,1.110785
4,-0.453793


In [23]:
# explore the timestamp column 
time_column = conn.execute(f"""
SELECT timestamp FROM '{DATA_PATH}'
""").df()
time_column.tail(20)

,timestamp
2009958,2022-02-24 04:21:00
2009959,2022-02-24 04:22:00
2009960,2022-02-24 04:23:00
2009961,2022-02-24 04:24:00
2009962,2022-02-24 04:25:00
2009963,2022-02-24 04:26:00
2009964,2022-02-24 04:27:00
2009965,2022-02-24 04:28:00
2009966,2022-02-24 04:29:00
2009967,2022-02-24 04:30:00


In [31]:
# explore the feed_rate column 
(pandas_dataframe['feed_rate'].isnull().sum() / len(parquet_dataframe)) * 100

np.float64(4.299450043731822)

In [35]:
feed_rate_count = conn.execute(f"""
SELECT batch_id, AVG(feed_rate) AS avg_feed_rate_per_batch
FROM '{DATA_PATH}' 
GROUP BY batch_id
ORDER BY batch_id ASC
""").df()
feed_rate_count


,batch_id,avg_feed_rate_per_batch
0,0,50.694217
1,1,49.138959
2,2,51.734478
3,3,49.222028
4,4,52.841332
...,...,...
11106,11106,50.193787
11107,11107,51.993010
11108,11108,47.144466
11109,11109,51.466394


In [36]:
feed_rate_shift = conn.execute(f"""
SELECT operator_shift, AVG(feed_rate) AS avg_feed_rate_per_batch
FROM '{DATA_PATH}' 
GROUP BY operator_shift
""").df()
feed_rate_shift

,operator_shift,avg_feed_rate_per_batch
0,Night,49.993510
1,b,49.932640
2,A,49.988836
3,N,49.974384
4,Shift A,49.981530
5,A,50.009526
6,a,49.997081
7,night,50.005371
8,Night,50.000540
9,B,49.947082


In [41]:
# check on the percentage purity 
feed_purity_check = conn.execute(f"""
SELECT batch_id, AVG(feed_purity) AS avg_feed_purity_per_batch
FROM '{DATA_PATH}' 
GROUP BY batch_id
ORDER BY batch_id ASC
""").df()
feed_purity_check.sample(20)

,batch_id,avg_feed_purity_per_batch
2567,2567,0.997595
7325,7325,0.960303
6500,6500,0.970787
1945,1945,0.965508
206,206,0.974958
1753,1753,96.901295
5020,5020,0.977430
10311,10311,0.980927
7131,7131,0.988669
6738,6738,0.962502


In [42]:
# investigate the ph column 
(pandas_dataframe['ph'].isnull().sum() / len(pandas_dataframe)) * 100

np.float64(4.243429530074459)

In [51]:
ph_values = conn.execute(f"""
SELECT batch_id, AVG(ph) as avg_ph_values FROM '{DATA_PATH}' GROUP BY batch_id
""").df()
print(ph_values.sample(20))

       batch_id  avg_ph_values
5704      11029       6.670651
2793      10484       7.070572
2028       6377       6.708303
249         936       6.621960
4985       7064       6.572342
9380       2530       6.865163
2561       4660       6.621273
10703      2362       6.981998
2847      10809       6.628967
10340      9177       6.868027
4863       6129       7.064949
7989       2798       7.059792
2661       5372       7.152187
3879       3838       6.751185
890        7931       6.740523
9838      10189       6.989516
2294       2502       6.754947
3252       8782       6.957703
3751       2713       7.051987
3645       7894       6.999304


In [60]:
# inspect the ambient humidty column
ambient_humidty_grouping = pandas_dataframe.groupby("batch_id")['ambient_humidity'].agg(['mean'])
print(ambient_humidty_grouping.sample(20))

               mean
batch_id           
3698      49.908438
6656      39.225424
7946      69.634149
4538      77.964801
346       63.772516
4226      67.948091
6745      32.862527
4270      71.454946
692       77.909532
2632      38.989176
3909      56.332265
7637      57.000917
7614      60.189991
749       73.068674
5386      57.890525
7837      75.416292
7979      66.886786
7190      39.591414
40        57.015375
11056     54.298157


In [62]:
# batch count query 
print(conn.execute(f"""SELECT COUNT(DISTINCT batch_id) as distinct_batches, 
MAX(batch_id) as max_id, 
MAX(batch_id)+1 as max_plus_1, 
COUNT(*) as total 
FROM '{DATA_PATH}'""").df())

   distinct_batches  max_id  max_plus_1    total
0             11111   11110       11111  2009978


In [72]:
batch_counts = conn.execute(f"""
SELECT batch_id, 
COUNT(*) AS counts_per_batch 
FROM '{DATA_PATH}' 
GROUP BY batch_id 
ORDER BY counts_per_batch
""").df()
batch_counts.to_csv("../docs/01_batch_counts.csv", index=False)

In [74]:
# get the count of the batches 
print(conn.execute(f"""
WITH get_batch_counts AS(
SELECT batch_id, 
COUNT(*) AS counts_per_batch 
FROM '{DATA_PATH}' 
GROUP BY batch_id 
ORDER BY counts_per_batch
)
SELECT COUNT(*)
FROM get_batch_counts
WHERE counts_per_batch = 180
""").df())


   count_star()
0          4477


In [80]:
# check the lot 
cleaned_catalyst_lot = conn.execute(f"""
SELECT DISTINCT(catalyst_lot) as unique_catalayst_lot FROM '{DATA_PATH}' 
""").df()
cleaned_catalyst_lot.sample(20)

,unique_catalayst_lot
69,l089
165,L090
207,l034
91,L045
164,l081
191,l031
135,L010
343,LOT-081
313,l040
317,LOT-019


In [ ]:
# lets clean the catalyst_lot column 
print(conn.execute(f"""
SELECT DISTINCT(catalyst_lot REPLACE('LOT-' || LPAD(REGEXP_EXTRACT('\d+)'), 3, '0')) as new_catalyst_lot
FROM '{DATA_PATH}'
"""))